In [ ]:
# 1. Garantir instalação do Sentence-Transformers
!pip install -q sentence-transformers

import pandas as pd
import re
import os
import logging
import warnings
from sentence_transformers import SentenceTransformer

# -------------------------------------------------------------------
# STEP 1: IMPORTAÇÃO DA PLANILHA GOOGLE SHEETS
# -------------------------------------------------------------------
sheet_id = "13knxQn9oHEWTp2xnmPdkWkaTJs_E4UlR8eiXDtsGLjc"
url = f"https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=csv"

df_original = pd.read_csv(url)
coluna_texto = df_original.iloc[:, 0].dropna().astype(str)

# -------------------------------------------------------------------
# STEP 2: EXTRAÇÃO E COMBINAÇÃO (Construct + Response Scale)
# -------------------------------------------------------------------
padrao_construct = r"Construct\s*/\s*Question:\s*([^\r\n]+)"
padrao_scale = r"Response\s*Scale:\s*([^\r\n]+)"

dados_extraidos = []
for texto in coluna_texto:
    texto_limpo = texto.replace('\xa0', ' ')
    constructs = re.findall(padrao_construct, texto_limpo, flags=re.IGNORECASE)
    scales = re.findall(padrao_scale, texto_limpo, flags=re.IGNORECASE)

    for c, s in zip(constructs, scales):
        c_clean = c.strip().strip('"').strip("'")
        s_clean = s.strip().strip('"').strip("'")
        dados_extraidos.append(f"Construct/Question: {c_clean} | Response Scale: {s_clean}")

df_combinado = pd.DataFrame(dados_extraidos, columns=["Construct_Response_Combinado"])
print(f" Planilha importada e {len(df_combinado)} linhas processadas!")


 Planilha importada e 278 linhas processadas!


In [ ]:
# Adiciona o código do item (V1, V2, V3...) mapeado para cada linha
df_combinado["Item"] = [f"V{i+1}" for i in range(len(df_combinado))]

# Salva o arquivo com os textos para o R ler
df_combinado.to_csv("tabela_textos_itens.csv", index=False)

print(" Tabela 'tabela_textos_itens.csv' criada para o R!")

 Tabela 'tabela_textos_itens.csv' criada para o R!


In [ ]:
# -------------------------------------------------------------------
# STEP 3: EMBEDDINGS SILENCIOSOS (MPNET)
# -------------------------------------------------------------------
warnings.filterwarnings("ignore")
logging.getLogger("sentence_transformers").setLevel(logging.ERROR)
logging.getLogger("transformers").setLevel(logging.ERROR)
os.environ["TOKENIZERS_PARALLELISM"] = "false"

model = SentenceTransformer("all-mpnet-base-v2")
embeddings = model.encode(
    df_combinado["Construct_Response_Combinado"].tolist(),
    show_progress_bar=False,
    convert_to_numpy=True
)

# -------------------------------------------------------------------
# STEP 4: SALVAR MATRIZ CSV PARA O R
# -------------------------------------------------------------------
df_matriz = pd.DataFrame(embeddings)
df_matriz.to_csv("matriz_embeddings.csv", index=False)

print(f" Embeddings regerados! Matriz final salva como 'matriz_embeddings.csv' ({embeddings.shape[0]} linhas x {embeddings.shape[1]} colunas).")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

 Embeddings regerados! Matriz final salva como 'matriz_embeddings.csv' (278 linhas x 768 colunas).


In [ ]:
!pip install -q cohere openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 357.0/357.0 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 64.0 MB/s eta 0:00:00


In [ ]:
import cohere
import pandas as pd
import numpy as np

# 1. Coloque a sua chave da Cohere aqui
COHERE_API_KEY = ""

co = cohere.Client(COHERE_API_KEY)

# 2. Pegar a lista de textos combinados da nossa coluna
textos = df_combinado["Construct_Response_Combinado"].tolist()

# 3. Gerar embeddings em lotes (máximo de 96 por requisição na Cohere)
batch_size = 96
all_cohere_embeddings = []

for i in range(0, len(textos), batch_size):
    lote = textos[i:i + batch_size]
    resposta = co.embed(
        texts=lote,
        model="embed-english-v3.0", # ou "embed-multilingual-v3.0"
        input_type="search_document"
    )
    all_cohere_embeddings.extend(resposta.embeddings)

matriz_cohere = np.array(all_cohere_embeddings)

# 4. Salvar CSV para usar no R
df_cohere = pd.DataFrame(matriz_cohere)
df_cohere.to_csv("matriz_embeddings_cohere.csv", index=False)

print(f" Embeddings Cohere gerados! Matriz salva como 'matriz_embeddings_cohere.csv' ({matriz_cohere.shape[0]} x {matriz_cohere.shape[1]})")

 Embeddings Cohere gerados! Matriz salva como 'matriz_embeddings_cohere.csv' (278 x 1024)


In [ ]:
from openai import OpenAI
import pandas as pd
import numpy as np

# 1. Coloque a sua chave da OpenAI aqui
OPENAI_API_KEY = ""

client = OpenAI(api_key=OPENAI_API_KEY)

# 2. Pegar a lista de textos combinados
textos = df_combinado["Construct_Response_Combinado"].tolist()

# 3. Gerar os embeddings usando text-embedding-3-small
resposta = client.embeddings.create(
    input=textos,
    model="text-embedding-3-small"
)

# 4. Extrair a matriz de vetores
embeddings_openai = [item.embedding for item in resposta.data]
matriz_openai = np.array(embeddings_openai)

# 5. Salvar CSV para usar no R
df_openai = pd.DataFrame(matriz_openai)
df_openai.to_csv("matriz_embeddings_openai.csv", index=False)

print(f" Embeddings OpenAI gerados! Matriz salva como 'matriz_embeddings_openai.csv' ({matriz_openai.shape[0]} x {matriz_openai.shape[1]})")

 Embeddings OpenAI gerados! Matriz salva como 'matriz_embeddings_openai.csv' (278 x 1536)


In [ ]:
import os
import logging
import warnings
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer

# 1. Silenciar logs e warnings do Hugging Face e PyTorch
warnings.filterwarnings("ignore")
logging.getLogger("sentence_transformers").setLevel(logging.ERROR)
logging.getLogger("transformers").setLevel(logging.ERROR)
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# 2. Carregar o modelo MiniLM Paraphrase BERT
model_minilm = SentenceTransformer("paraphrase-MiniLM-L6-v2")

# 3. Pegar os textos combinados
textos = df_combinado["Construct_Response_Combinado"].tolist()

# 4. Gerar os embeddings de forma 100% silenciosa
embeddings_minilm = model_minilm.encode(
    textos,
    show_progress_bar=False,
    convert_to_numpy=True
)

# 5. Salvar CSV para usar no R
df_minilm = pd.DataFrame(embeddings_minilm)
df_minilm.to_csv("matriz_embeddings_minilm.csv", index=False)

print(f" Embeddings MiniLM Paraphrase BERT gerados em silêncio!")
print(f" Matriz salva como 'matriz_embeddings_minilm.csv' ({embeddings_minilm.shape[0]} linhas x {embeddings_minilm.shape[1]} colunas)")

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.51k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

 Embeddings MiniLM Paraphrase BERT gerados em silêncio!
 Matriz salva como 'matriz_embeddings_minilm.csv' (278 linhas x 384 colunas)
